# Notebook 2 — Online Parameter Estimation

**Project:** Adaptive Execution via Online Parameter Estimation  
**Author:** Changkui Wu (FSU Financial Mathematics PhD, 2026)

---

## Overview

This notebook applies the online estimator from the PhD thesis to the real
AAPL LOBSTER data, demonstrating that the resilience parameter $r$ can be
learned in real time from the observed LOB deviation process $D_t$.

### The Estimator (Thesis Eq. 3.2)

Given the state equation $dD_t = -r^* D_t \, dt + \sigma \, dB_t$, the
likelihood-inspired online estimator is:

$$d\hat{r}_t = \eta_t \underbrace{\frac{\partial f}{\partial r}^\top}_{= -D_t} \Sigma^{-1} \underbrace{\left[f(D_t, r^*) - f(D_t, \hat{r}_t)\right]}_{\text{drift error}} dt + \text{martingale}$$

Discretised (Euler–Maruyama):

$$\hat{r}_{n+1} = \hat{r}_n + \eta_n \cdot (-D_n) \cdot \sigma^{-2} \cdot \underbrace{\left(\frac{\Delta D_n}{\Delta t} + \hat{r}_n D_n\right)}_{\text{innovation}} \cdot \Delta t$$

**Theorem 4.3 (thesis):** Under standard regularity, monotonicity, and
learning-rate conditions, $\hat{r}_t \to r^*$ almost surely as $t \to \infty$.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from lobster import load_lobster, resample, estimate_params
from estimator import run_online_estimator, converged_estimate

DARK, GRID = '#0f1117', '#1a1d2e'
TEXT, BLUE  = '#e0e0e0', '#4a9eff'
GREEN, ORANGE, RED = '#50fa7b', '#ffb86c', '#ff5555'
PURPLE, CYAN, YELLOW = '#bd93f9', '#8be9fd', '#f1fa8c'

plt.rcParams.update({
    'figure.facecolor': DARK, 'axes.facecolor': GRID,
    'axes.edgecolor': '#333355', 'axes.labelcolor': TEXT,
    'xtick.color': TEXT, 'ytick.color': TEXT,
    'text.color': TEXT, 'grid.color': '#252540',
    'grid.linewidth': 0.6, 'legend.facecolor': GRID,
    'legend.labelcolor': TEXT, 'font.size': 10,
})

DATA_DIR = os.path.join('..', 'data')
MSG_FILE = os.path.join(DATA_DIR, 'AAPL_2012-06-21_34200000_57600000_message_5.csv')
OBK_FILE = os.path.join(DATA_DIR, 'AAPL_2012-06-21_34200000_57600000_orderbook_5.csv')

lob    = load_lobster(MSG_FILE, OBK_FILE)
params = estimate_params(lob)
DT     = 1.0   # resample step (seconds)

t_grid, Dt_g, mid_g, spr_g, q_g = resample(lob, dt=DT)
sigma = params['sigma']
r_mle = params['r']

print(f"Calibrated parameters:")
print(f"  r_MLE  = {r_mle:.4f} /s   (half-life {np.log(2)/r_mle:.1f}s)")
print(f"  sigma  = {sigma:.5f} $/sqrt(s)")
print(f"  Resampled grid: {len(t_grid):,} points at dt={DT}s")


## 1. Estimator Convergence from Multiple Starting Points

A key test of the thesis's almost-sure convergence guarantee: starting from
very different initial guesses $\hat{r}_0$, the estimator should converge
to the same value $r^*$ regardless of initialisation.

This mirrors **Figure 5.1** in the thesis.


In [ ]:
r0_list   = [0.001, 0.01, 0.1, 0.5, 2.0, 5.0]
colors    = [CYAN, BLUE, GREEN, ORANGE, RED, PURPLE]
t_since   = (t_grid - t_grid[0]) / 3600.0   # hours

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Online Estimator Convergence — Multiple Initialisations',
             color=TEXT, fontsize=12)

for r0, col in zip(r0_list, colors):
    rh = run_online_estimator(Dt_g, DT, sigma, r0=r0, eta0=0.3, alpha=0.75)
    for ax, logy in zip(axes, [False, True]):
        ax.plot(t_since, rh, color=col, lw=1.0, alpha=0.85, label=f'$r_0$={r0}')

for ax, logy in zip(axes, [False, True]):
    ax.axhline(r_mle, color=RED, lw=1.8, ls='--', alpha=0.7,
               label=f'MLE = {r_mle:.4f}')
    ax.set_xlabel('Time since open (hours)')
    ax.set_ylabel('$\hat{r}_t$ (per second)')
    ax.set_xlim(0, t_since[-1])
    ax.legend(fontsize=8, ncol=2)
    ax.set_facecolor(GRID)

axes[0].set_title('Linear scale', color=TEXT)
axes[1].set_title('Log scale', color=TEXT)
axes[1].set_yscale('log')
axes[1].set_ylim(1e-4, 20)

plt.tight_layout()
plt.savefig(os.path.join('..','outputs','nb2_convergence.png'),
            dpi=130, bbox_inches='tight', facecolor=DARK)
plt.show()


## 2. Effect of Learning Rate Schedule

The learning rate $\eta_t = \eta_0 / (1+t)^\alpha$ controls the
speed–accuracy trade-off.

- **Large $\alpha$** (close to 1): slower decay, suppresses noise better asymptotically
- **Small $\alpha$** (close to 0.5): faster early convergence, more noise later

This replicates the analysis in **Figure 5.3** of the thesis on real data.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Effect of Learning Rate Schedule  $\eta_t = \eta_0/(1+t)^\alpha$',
             color=TEXT, fontsize=12)

alpha_list  = [0.55, 0.65, 0.75, 0.85, 0.95]
alpha_cols  = [CYAN, BLUE, GREEN, ORANGE, RED]

for al, col in zip(alpha_list, alpha_cols):
    rh = run_online_estimator(Dt_g, DT, sigma, r0=2.0, eta0=0.3, alpha=al)
    err = np.abs(rh - r_mle)
    for ax, y in zip(axes, [rh, err]):
        ax.plot(t_since, y, color=col, lw=1.0, alpha=0.85, label=f'α={al}')

axes[0].axhline(r_mle, color=RED, lw=1.5, ls='--', label=f'MLE={r_mle:.4f}')
axes[0].set_ylabel('$\hat{r}_t$'); axes[0].set_title('Estimate trajectory')
axes[1].set_ylabel('$|\hat{r}_t - r_{MLE}|$'); axes[1].set_title('Absolute error')
axes[1].set_yscale('log')

for ax in axes:
    ax.set_xlabel('Hours since open')
    ax.set_xlim(0, t_since[-1])
    ax.legend(fontsize=9)
    ax.set_facecolor(GRID)

plt.tight_layout()
plt.savefig(os.path.join('..','outputs','nb2_learning_rate.png'),
            dpi=130, bbox_inches='tight', facecolor=DARK)
plt.show()


## 3. Three-Term Lyapunov Decomposition (Thesis Chapter 4)

The convergence proof decomposes the Lyapunov function
$V_t = \frac{1}{2}\|\hat{r}_t - r^*\|^2$ into three terms:

$$V_t \leq \underbrace{E_t^{-1} V_0}_{\text{(I) initial condition}} +
           \underbrace{E_t^{-1} C \int_0^t E_s \eta_s^2 \, ds}_{\text{(II) diffusion remainder}} +
           \underbrace{E_t^{-1} \int_0^t E_s \, dM_s}_{\text{(III) martingale}}$$

where $E_t = \exp\!\left(2\lambda_0 \int_0^t \eta_s \, ds\right)$.

All three terms vanish as $t \to \infty$, giving almost-sure convergence.
Below we track them numerically on the real LOBSTER data — matching
**Figure 5.5** from the thesis.


In [ ]:
alpha0, eta0_val = 0.75, 0.3
lam0 = params['r']    # use MLE as proxy for lambda_0 in the decomposition

# Compute the three terms
rh  = run_online_estimator(Dt_g, DT, sigma, r0=2.0, eta0=eta0_val, alpha=alpha0)
V   = 0.5 * (rh - r_mle)**2   # Lyapunov function

# Integrating factor E_t
S   = np.cumsum(eta0_val / (1 + np.arange(len(t_grid)) * DT)**alpha0) * DT
Et  = np.exp(2 * lam0 * S)

# Term I: E_t^{-1} * V(0)
term_I = V[0] / Et

# Term II: E_t^{-1} * C * integral E_s eta_s^2 ds
eta_sq = (eta0_val / (1 + np.arange(len(t_grid)) * DT)**alpha0)**2
integ  = np.cumsum(Et * eta_sq) * DT
term_II = integ / Et

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Three-Term Lyapunov Decomposition  (Thesis Chapter 4, Fig 5.5)',
             color=TEXT, fontsize=12)

S_plot = S   # effective time

for ax, logy in zip(axes, [False, True]):
    ax.plot(S_plot, V,        color=CYAN,   lw=1.0, label='$V_t = \frac{1}{2}|\hat{r}_t - r^*|^2$')
    ax.plot(S_plot, term_I,   color=ORANGE, lw=1.5, ls='--', label='Term (I): initial condition')
    ax.plot(S_plot, term_II,  color=GREEN,  lw=1.5, ls=':',  label='Term (II): diffusion remainder')
    ax.set_xlabel('Effective time $S(t) = \int_0^t \eta_s \, ds$')
    ax.set_ylabel('Value')
    ax.legend(fontsize=9)
    ax.set_facecolor(GRID)
    if logy:
        ax.set_yscale('log')
        ax.set_ylim(1e-8, V[0]*2)
        ax.set_title('Log scale')
    else:
        ax.set_title('Linear scale')

plt.tight_layout()
plt.savefig(os.path.join('..','outputs','nb2_lyapunov.png'),
            dpi=130, bbox_inches='tight', facecolor=DARK)
plt.show()
print("Term (I)  converges to 0:", f"{term_I[-1]:.2e}")
print("Term (II) converges to 0:", f"{term_II[-1]:.2e}")


## 4. Benchmark Comparison: Online vs MLE

The online estimator is a **sequential** algorithm — it processes one observation
at a time and can be used in real time.  The MLE uses all data simultaneously.

Key questions:
- How quickly does the online estimator approach the MLE?
- What is the price of online (sequential) estimation?


In [ ]:
# Compare online vs MLE at different time horizons
horizons = np.linspace(0.1, 1.0, 20)   # fraction of trading day
r_online_at_t = []
r_mle_at_t    = []

for frac in horizons:
    n_end = int(frac * len(Dt_g))
    # Online estimator at this fraction
    rh_partial = run_online_estimator(Dt_g[:n_end], DT, sigma, r0=2.0,
                                      eta0=0.3, alpha=0.75)
    r_online_at_t.append(converged_estimate(rh_partial, burn_in=0.5))
    # MLE on this partial data
    dDt   = np.diff(Dt_g[:n_end])
    Dn    = Dt_g[:n_end-1]
    r_mle_here = -np.dot(dDt, Dn*DT) / (np.dot(Dn,Dn)*DT**2 + 1e-30)
    r_mle_at_t.append(max(0.001, float(r_mle_here)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.set_facecolor(GRID)
ax.plot(horizons * (t_grid[-1]-t_grid[0])/3600, r_online_at_t,
        color=CYAN, lw=2.0, marker='o', ms=4, label='Online estimator (converged half)')
ax.plot(horizons * (t_grid[-1]-t_grid[0])/3600, r_mle_at_t,
        color=ORANGE, lw=2.0, marker='s', ms=4, label='MLE (batch, uses all data to t)')
ax.axhline(r_mle, color=RED, lw=1.5, ls='--', label=f'Full-day MLE = {r_mle:.4f}')
ax.set_xlabel('Hours of data used')
ax.set_ylabel('Estimated $r$ (per second)')
ax.set_title('Online vs MLE Estimates as Data Accumulates', color=TEXT)
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join('..','outputs','nb2_online_vs_mle.png'),
            dpi=130, bbox_inches='tight', facecolor=DARK)
plt.show()

print("Full-day MLE:          r = ", f"{r_mle:.4f}")
print("Online (full day):     r = ", f"{converged_estimate(rh, burn_in=0.5):.4f}")
print()
print("The online estimator converges to the MLE in the long run,")
print("but provides a REAL-TIME estimate available at every tick.")


## Summary

| Property | Online Estimator | MLE |
|----------|-----------------|-----|
| Real-time updates | ✅ | ❌ |
| Convergence guarantee | ✅ a.s. (Theorem 4.3) | ✅ consistency |
| Works under adaptive feedback | ✅ (no stationarity needed) | ⚠️ assumes stationarity |
| Computational cost | O(1) per step | O(N) batch |

**Key takeaway:** The online estimator provides real-time parameter estimates
that are theoretically guaranteed to converge — making it suitable for use
inside an adaptive execution loop, which is exactly what Notebook 3 demonstrates.
